# 02 - Ingeniería de Variables

## Predicción del riesgo de rotura de stock en un entorno logístico

Este notebook construye las variables necesarias para el modelo predictivo a partir del dataset original de ventas.  
El objetivo es generar variables temporales, variables de comportamiento histórico de la demanda y una variable objetivo binaria (`stockout_risk`) que represente el riesgo de rotura de stock.

> Nota metodológica: las variables `stock_estimated`, `safety_stock` y `future_demand_7d` se utilizan para construir la variable objetivo. Por tanto, **no deben utilizarse como variables predictoras en el entrenamiento del modelo**, ya que producirían leakage.


## 1. Carga de librerías y datos

Se carga el dataset original y se ordena por tienda, producto y fecha.  
Este orden es imprescindible para calcular correctamente lags, ventanas móviles y demanda futura por cada combinación `store_id` - `item_id`.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW_DATA_PATH = "../data/raw/retail_sales.csv"
PROCESSED_DATA_PATH = "../data/processed/stockout_dataset.csv"

df = pd.read_csv(RAW_DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

print("Shape inicial:", df.shape)
df.head()


## 2. Revisión inicial del dataset

Antes de crear nuevas variables se revisa la estructura básica del dataset, los tipos de datos y la existencia de nulos o duplicados.

In [ ]:
display(df.info())

print("\nNulos por columna:")
display(df.isna().sum())

print("\nDuplicados:", df.duplicated().sum())

display(df.describe(include="all").T)

## 3. Variables temporales

Se generan variables de calendario que pueden capturar patrones estacionales y ciclos de demanda:

- Año.
- Mes.
- Trimestre.
- Semana del año.
- Día del mes.
- Día de la semana.

Estas variables son relevantes porque la demanda puede variar según el momento del año, el mes o el día de la semana.

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday

df[["date", "year", "month", "quarter", "week", "day", "weekday"]].head()

## 4. Variables históricas de demanda: lags

Se crean variables de ventas pasadas para cada combinación tienda-producto:

- Ventas hace 7 días.
- Ventas hace 14 días.
- Ventas hace 30 días.

Estas variables permiten incorporar el comportamiento histórico reciente sin utilizar información futura.

In [ ]:
group_cols = ["store_id", "item_id"]

df["sales_lag_7"] = (
    df.groupby(group_cols)["sales"]
      .shift(7)
)

df["sales_lag_14"] = (
    df.groupby(group_cols)["sales"]
      .shift(14)
)

df["sales_lag_30"] = (
    df.groupby(group_cols)["sales"]
      .shift(30)
)

df[["store_id", "item_id", "date", "sales", "sales_lag_7", "sales_lag_14", "sales_lag_30"]].head(35)
